# Dataset Manifest Generator and Prepare Dataset

This notebook generates a manifest for the full train dataset. Each row maps one image to its label, batch, and split. Labels support one or more percentages, for example `R5` or `R5 S10` (rice hull 5%, salt 10%) and then prepares and preprocess the dataset for it to stay in the disk for future retraining.

Keep the same output keys throughout one training run. For rice hull and salt, use `R5 S0` when salt is absent. Output keys are sorted consistently, so `R5 S10` and `S10 R5` both map to `[5, 10]` in `R, S` order. Keep the original folder label unchanged for prepared-image paths.

## Manifest format

| image_path                      | label | batch   | split      |
|:--------------------------------|:-----:|:--------|:----------:|
| dataset/R5/batch_1/img_001.jpg  | R5    | batch_1 | training   |
| dataset/R0/batch_2/img_001.jpg  | R0    | batch_2 | testing    |
| dataset/R10/batch_3/img_001.jpg | R10   | batch_3 | validation |

In [1]:

import pandas as pd
from utils.labels import infer_label_keys
from utils.prepare_dataset import prepare_dataset
import config

config.check_config(is_notebook=True)

CHECK CONFIGURATION DETAILS:
RAW DATASET DIR: C:\Users\realb\Documents\PROJECT_FILES\Dataset\soybean_meal
TASK: regression
MANIFEST FILENAME (same dir as this file): C:\Users\realb\Documents\PROJECT_FILES\DABARResearchTraining\soybean_meal_regression.csv
PREPROCESSED DATASET DIR: C:\Users\realb\Documents\PROJECT_FILES\Dataset\for_training\regression\soybean_meal_augmented
PTH MODEL OUTPUT: C:\Users\realb\Documents\PROJECT_FILES\DABARResearchTraining\models\pth\soybean_meal.pth
ONNX MODEL OUTPUT: C:\Users\realb\Documents\PROJECT_FILES\DABARResearchTraining\models\onnx\soybean_meal.onnx
EPOCHS: 300
BATCH SIZE: 32
Review the configuration above before running the next cell.


Records the ```image_path```, ```label```, and ```batch```

In [4]:
records = []

for image in config.RAW_DATASET_DIR.rglob("*"):
    if not image.is_file():
        continue
    if image.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue

    records.append({
        "image_path": str(image),
        "batch": image.parent.name,
        "label": image.parent.parent.name,
    })
print(records[:5])
if not records:
    raise ValueError("IMAGE WHEREE???????")

df = pd.DataFrame(records)

[{'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\soybean_meal\\L5\\batch_1\\img_20261015_135856_0000.jpg', 'batch': 'batch_1', 'label': 'L5'}, {'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\soybean_meal\\L5\\batch_1\\img_20261015_141933_0000.jpg', 'batch': 'batch_1', 'label': 'L5'}, {'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\soybean_meal\\L5\\batch_1\\img_20261015_142152_0000.jpg', 'batch': 'batch_1', 'label': 'L5'}, {'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\soybean_meal\\L5\\batch_1\\img_20261015_142434_0000.jpg', 'batch': 'batch_1', 'label': 'L5'}, {'image_path': 'C:\\Users\\realb\\Documents\\PROJECT_FILES\\Dataset\\soybean_meal\\L5\\batch_1\\img_20261015_142448_0000.jpg', 'batch': 'batch_1', 'label': 'L5'}]


Splits the batches in their ```split``` calculated 70% training 15% testing 15% validation, splitted by batch count, not image count

In [5]:
if config.TASK == "classification":
    # Keep the complete folder names as classes.
    class_names = sorted(df["label"].unique())
    print("Classes:", class_names)
else:
    # Parse labels such as R5 S0 into regression outputs.
    label_keys = infer_label_keys(df["label"])
    print("Output order:", label_keys)
    
print(df.head())

Output order: ('L',)
                                          image_path    batch label
0  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1    L5
1  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1    L5
2  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1    L5
3  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1    L5
4  C:\Users\realb\Documents\PROJECT_FILES\Dataset...  batch_1    L5


In [6]:
batches = df[["label", "batch"]].drop_duplicates().copy()
batches["split"] = "" # split are training, validation, and testing

for label in batches["label"].unique():
    mask = batches["label"] == label 
    group = batches[mask].sample(frac=1, random_state=42)
    
    batches_per_label = len(group)
    
    if batches_per_label < 3:
        raise ValueError(f"Not enough batches for label '{label}' to split into training, validation, and testing.")
    
    train_batches = min(round(batches_per_label * 0.7), batches_per_label - 2)
    remaining_batches = batches_per_label - train_batches
    validation_batches = remaining_batches // 2
    test_batches = remaining_batches - validation_batches
    
    print(f"Label: {label}, Total Batches: {batches_per_label}, Train: {train_batches}, Validation: {validation_batches}, Test: {test_batches}")
    
    batches.loc[group.index, "split"] = (
        ["training"] * train_batches
        + ["validation"] * validation_batches
        + ["testing"] * test_batches
    )
    
df = df.merge(batches, on=["label", "batch"], validate="many_to_one")

Label: L5, Total Batches: 10, Train: 7, Validation: 1, Test: 2
Label: L10, Total Batches: 10, Train: 7, Validation: 1, Test: 2


In [7]:
print(pd.crosstab(batches["label"], batches["split"]))
print(pd.crosstab(df["label"], df["split"]))

print(config.MANIFEST_FILENAME)
df.to_csv(config.MANIFEST_FILENAME, index=False)

split  testing  training  validation
label                               
L10          2         7           1
L5           2         7           1
split  testing  training  validation
label                               
L10         20        70          10
L5          20        70          10
C:\Users\realb\Documents\PROJECT_FILES\DABARResearchTraining\soybean_meal_regression.csv


### Prepare and preprocess dataset

In [8]:
prepare_dataset(config.MANIFEST_FILENAME, config.PREPROCESSED_DATASET_DIR)

1/200: PROCESSED img_20261015_135856_0000
1/200: PROCESSED img_20261015_135856_0000_augment_contrast
1/200: PROCESSED img_20261015_135856_0000_augment_gamma
1/200: PROCESSED img_20261015_135856_0000_augment_saturation
2/200: PROCESSED img_20261015_141933_0000
2/200: PROCESSED img_20261015_141933_0000_augment_contrast
2/200: PROCESSED img_20261015_141933_0000_augment_gamma
2/200: PROCESSED img_20261015_141933_0000_augment_saturation
3/200: PROCESSED img_20261015_142152_0000
3/200: PROCESSED img_20261015_142152_0000_augment_contrast
3/200: PROCESSED img_20261015_142152_0000_augment_gamma
3/200: PROCESSED img_20261015_142152_0000_augment_saturation
4/200: PROCESSED img_20261015_142434_0000
4/200: PROCESSED img_20261015_142434_0000_augment_contrast
4/200: PROCESSED img_20261015_142434_0000_augment_gamma
4/200: PROCESSED img_20261015_142434_0000_augment_saturation
5/200: PROCESSED img_20261015_142448_0000
5/200: PROCESSED img_20261015_142448_0000_augment_contrast
5/200: PROCESSED img_202610